In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
# ============================================================================
# 第 1 步：数据准备 —— 加载原始数据，了解结构
# ============================================================================

df_customer = pd.read_csv("Mall_Customers.csv")
df_member = pd.read_csv("member_info.csv")

print("【主表】Mall_Customers.csv  形状(行,列):", df_customer.shape)
print(df_customer.head())
print("\n字段信息：")
print(df_customer.info())

print("\n【附表】member_info.csv  形状(行,列):", df_member.shape)
print(df_member.head())

# 两表主键是否对齐（为后续数据融合做准备）
overlap = len(set(df_customer["CustomerID"]) & set(df_member["CustomerID"]))
print(f"\n两表 CustomerID 对齐数量: {overlap} / {len(df_customer)}")

【主表】Mall_Customers.csv  形状(行,列): (200, 5)
   CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)
0           1    Male   19                  15                      39
1           2    Male   21                  15                      81
2           3  Female   20                  16                       6
3           4  Female   23                  16                      77
4           5  Female   31                  17                      40

字段信息：
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   CustomerID              200 non-null    int64
 1   Gender                  200 non-null    str  
 2   Age                     200 non-null    int64
 3   Annual Income (k$)      200 non-null    int64
 4   Spending Score (1-100)  200 non-null    int64
dtypes: int64(4), str(1)
memory usage: 8.9 KB
None

【附表】member_info.csv  形状

In [2]:
# ============================================================================
# 第 2 步：数据清洗 —— 处理重复、缺失、异常值
# ============================================================================

# 人为构造一份带脏数据（重复/缺失/异常）的副本用于演示。
df_dirty = df_customer.copy()
df_dirty = pd.concat([df_dirty, df_customer.iloc[[0, 1]]], ignore_index=True)  # 加2行重复
df_dirty.loc[5, "Age"] = np.nan                 # 制造缺失
df_dirty.loc[10, "Annual Income (k$)"] = 999    # 制造异常值（正常范围应 <200）
print("构造脏数据后行数:", len(df_dirty), "（比原始多2行重复）")
# print("查看前15行数据",df_dirty.head(15))


# (1) 处理重复值：按 CustomerID 去重
print("\n重复行数:", df_dirty.duplicated(subset="CustomerID").sum())
df_dirty = df_dirty.drop_duplicates(subset="CustomerID")
print("去重后行数:", len(df_dirty))

# (2) 处理缺失值：数值列用中位数填充（中位数比均值更抗异常）
print("\n缺失值统计：")
print(df_dirty.isnull().sum())
df_dirty["Age"] = df_dirty["Age"].fillna(df_dirty["Age"].median())
print("Age 缺失值已用中位数填充。")

# (3) 处理异常值：IQR 箱线图法检测 + 盖帽法（clip 截断，不删样本）
def detect_and_clip(series, col_name):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = int(((series < lower) | (series > upper)).sum())
    print(f"  {col_name}: 检测到 {n_out} 个离群点，区间 [{lower:.2f}, {upper:.2f}]")
    return np.clip(series, lower, upper), n_out

for col in ["Annual Income (k$)", "Spending Score (1-100)"]:
    df_dirty[col], n = detect_and_clip(df_dirty[col], col)

# 展示被盖帽修正的异常收入
print("\n清洗后年收入统计：min={:.1f}, max={:.1f}（异常值 999 已被压到上限）".format(
    df_dirty["Annual Income (k$)"].min(), df_dirty["Annual Income (k$)"].max()))

构造脏数据后行数: 202 （比原始多2行重复）

重复行数: 2
去重后行数: 200

缺失值统计：
CustomerID                0
Gender                    0
Age                       1
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64
Age 缺失值已用中位数填充。
  Annual Income (k$): 检测到 3 个离群点，区间 [-12.00, 132.00]
  Spending Score (1-100): 检测到 0 个离群点，区间 [-22.62, 130.38]

清洗后年收入统计：min=15.0, max=132.0（异常值 999 已被压到上限）


In [3]:
# ============================================================================
# 第 3 步：数据融合 —— 两张表按 CustomerID 合并
# ============================================================================

df_all = pd.merge(df_dirty, df_member, on="CustomerID", how="inner")
print("合并后形状(行,列):", df_all.shape)
print("合并后字段：", list(df_all.columns))

合并后形状(行,列): (200, 7)
合并后字段： ['CustomerID', 'Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'RegisterDate', 'MemberLevel']


In [4]:
# ============================================================================
# 第 4 步：特征提取 —— 从原始字段生成新的有用特征
# ============================================================================

# 会员年限：由注册日期算出（特征提取：从旧字段算新字段）
df_all["RegisterDate"] = pd.to_datetime(df_all["RegisterDate"])
df_all["MemberYears"] = (pd.Timestamp.now() - df_all["RegisterDate"]).dt.days / 365.0

# 年龄段：把连续年龄离散化为类别（特征提取）
def age_group(age):
    if age < 30:
        return "青年"
    elif age < 50:
        return "中年"
    else:
        return "老年"

df_all["AgeGroup"] = df_all["Age"].apply(age_group)

# 收入/年龄比：综合两个字段的新指标（特征提取）
df_all["IncomePerAge"] = df_all["Annual Income (k$)"] / df_all["Age"].replace(0, np.nan)

print("新增特征：MemberYears(会员年限)、AgeGroup(年龄段)、IncomePerAge(收入年龄比)")
print(df_all[["CustomerID", "Age", "RegisterDate", "MemberYears", "AgeGroup", "IncomePerAge"]].head())

新增特征：MemberYears(会员年限)、AgeGroup(年龄段)、IncomePerAge(收入年龄比)
   CustomerID   Age RegisterDate  MemberYears AgeGroup  IncomePerAge
0           1  19.0   2015-09-24    11.000000       青年      0.789474
1           2  21.0   2015-11-29    10.819178       青年      0.714286
2           3  20.0   2017-10-30     8.898630       青年      0.800000
3           4  23.0   2019-02-24     7.578082       青年      0.695652
4           5  31.0   2023-02-12     3.608219       中年      0.548387


In [5]:
# ============================================================================
# 第 5 步：数据变换 —— 类别文字 -> 数字编码
# ============================================================================
# Gender、MemberLevel 为文字类别，转成数值
df_all = pd.get_dummies(df_all, columns=["Gender", "MemberLevel"], prefix=["Gender", "Level"])
print("类别列已独热编码。新增编码列：")
print([c for c in df_all.columns if c.startswith("Gender_") or c.startswith("Level_")])

类别列已独热编码。新增编码列：
['Gender_Female', 'Gender_Male', 'Level_普通会员', 'Level_金卡会员', 'Level_银卡会员']


In [6]:
# ============================================================================
# 第 6 步：数据转换 —— 数值归一化（缩放到 0~1）
# ============================================================================
num_cols = ["Age", "Annual Income (k$)", "Spending Score (1-100)", "MemberYears", "IncomePerAge"]
scaler = MinMaxScaler()
df_all[num_cols] = scaler.fit_transform(df_all[num_cols])

print("数值列已 MinMax 归一化到 [0,1]：", num_cols)
print("归一化后各列范围：")
print(df_all[num_cols].min().round(2).to_string(), "\n", df_all[num_cols].max().round(2).to_string())

数值列已 MinMax 归一化到 [0,1]： ['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'MemberYears', 'IncomePerAge']
归一化后各列范围：
Age                       0.0
Annual Income (k$)        0.0
Spending Score (1-100)    0.0
MemberYears               0.0
IncomePerAge              0.0 
 Age                       1.0
Annual Income (k$)        1.0
Spending Score (1-100)    1.0
MemberYears               1.0
IncomePerAge              1.0


In [7]:
# ============================================================================
# 结果输出与保存
# ============================================================================
df_final = df_all.drop(columns=["RegisterDate"])  # 日期已不再需要
df_final.to_csv("Mall_Customers_preprocessed.csv", index=False)

print("最终数据集形状(行,列):", df_final.shape)
print("已保存为：Mall_Customers_preprocessed.csv")
print("\n最终字段列表：")
for c in df_final.columns:
    print("  -", c)

最终数据集形状(行,列): (200, 12)
已保存为：Mall_Customers_preprocessed.csv

最终字段列表：
  - CustomerID
  - Age
  - Annual Income (k$)
  - Spending Score (1-100)
  - MemberYears
  - AgeGroup
  - IncomePerAge
  - Gender_Female
  - Gender_Male
  - Level_普通会员
  - Level_金卡会员
  - Level_银卡会员
